# From OSI to SLayer

**TL;DR:** SLayer can ingest an [OSI](https://open-semantic-interchange.org/) (Open Semantic Interchange) config — its `datasets`, `relationships`, and `metrics` — and turn it into queryable SLayer models. This notebook imports a small retail OSI config end-to-end and answers five questions, checking each answer against gold SQL.

It runs as four explicit steps (everything lands in a gitignored `.cache/` next to this notebook):

1. **Build the data** — create a tiny retail DuckDB (orders, customers, products, regions) with deterministic rows.
2. **Reference answers** — run the gold SQL *up front*, before the importer touches the file.
3. **Import** — convert `shop.osi.yaml` into SLayer models with `OsiToSlayerConverter`.
4. **Query** — point a SLayer client at the imported models and verify against gold.

> Unlike the [dbt MetricFlow demo](../11_dbt_metricflow/dbt_metricflow_nb.ipynb), this one is **fully offline** — no network access is needed at any point.

The committed OSI config ([`shop.osi.yaml`](shop.osi.yaml)) exercises every OSI feature the importer understands: `ai_context` metadata, `custom_extensions`, relationships that become joins, and simple / derived / cross-dataset / multi-hop metrics.

## Step 1 — Build the demo database

Create the retail DuckDB the OSI config binds to. OSI carries no column *types* — the importer reads them by **live introspection** of this database — so it has to exist first.

In [1]:
import os
import sys

import pandas as pd

# The setup helper lives next to this notebook.
sys.path.insert(0, os.getcwd())

from setup_osi import (
    build_shop_duckdb,
    convert_osi_to_slayer,
    compute_gold,
    OSI_CONFIG,
    DB_PATH,
    MODELS_DIR,
    DATASOURCE_NAME,
)
from slayer.async_utils import run_sync
from slayer.client.slayer_client import SlayerClient
from slayer.storage.yaml_storage import YAMLStorage

db_path = build_shop_duckdb(DB_PATH)
print(f"Built retail DuckDB '{db_path.name}' with orders, customers, products, regions.")

Built retail DuckDB 'shop.duckdb' with orders, customers, products, regions.


## Step 2 — Reference answers (gold SQL)

We run the gold SQL **first**, up front, and stash the expected numbers.

Why up front? The importer introspects the datasource live, opening a read-write SQLAlchemy engine on the DuckDB file — and DuckDB won't let a second raw connection share the file under a different configuration. So every gold query runs *before* the import (Step 3) touches the file.

In [2]:
# The reference answers live in setup_osi.compute_gold — shared with the
# agent-workflow notebook so both demos check against identical numbers.
GOLD = compute_gold(DB_PATH)

print("gold total amount :", GOLD["total"])
print("gold aov          :", GOLD["aov"])
print("gold cust_reach   :", GOLD["cust_reach"])
print("gold rev_plus_pop :", GOLD["rev_plus_pop"])
pd.DataFrame(GOLD["by_region"])

gold total amount : 1400.0
gold aov          : 233.33333333333334
gold cust_reach   : 466.6666666666667
gold rev_plus_pop : 4400.0


,region,amount
0,North,750.0
1,South,650.0


## Step 3 — Import the OSI config

This is the heart of the demo. `convert_osi_to_slayer` registers the DuckDB as a SLayer datasource, parses `shop.osi.yaml`, and runs [`OsiToSlayerConverter`](../../osi/osi_import.md):

- each OSI **dataset** becomes a SLayer model (column *types* from live introspection),
- each **relationship** becomes a LEFT join,
- each **metric** folds into a `ModelMeasure` formula on the model the converter picks as its anchor.

The converter reports anything it can't express cleanly — here, everything converts.

In [3]:
result = convert_osi_to_slayer(OSI_CONFIG, MODELS_DIR, DB_PATH)

print("Imported models:", [m.name for m in result.models])
unconverted, dropped = result.tally()
print(f"{unconverted} metrics unconverted, {dropped} items dropped")
print()
print(result.render_report())

Imported models: ['orders', 'customers', 'products', 'regions']
0 metrics unconverted, 0 items dropped

No conversion issues.


## Step 4 — The imported metadata survived

OSI's `ai_context` and `custom_extensions` don't just decorate the source file — the importer carries them onto the SLayer entities so agents can read them. Below, the `orders.amount` column keeps its label, its `ai_context` instructions as a description, and both blocks in `meta`; the `orders -> customers` join keeps its relationship `ai_context` as a description.

In [4]:
storage = YAMLStorage(base_dir=str(MODELS_DIR))
orders = run_sync(storage.get_model("orders", data_source=DATASOURCE_NAME))

amount = next(c for c in orders.columns if c.name == "amount")
print("orders.amount.label       :", amount.label)
print("orders.amount.description :", amount.description)
print("orders.amount.meta        :", amount.meta)
print()
for j in orders.joins:
    print(f"join orders -> {j.target_model}: {j.description!r}")
print()
print("measures on orders:", [m.name for m in orders.measures])

orders.amount.label       : Order amount
orders.amount.description : Gross order value in USD.
Synonyms: revenue, gross
orders.amount.meta        : {'osi_ai_context': {'instructions': 'Gross order value in USD.', 'synonyms': ['revenue', 'gross']}, 'osi_custom_extensions': [{'vendor_name': 'SNOWFLAKE', 'data': '{"unit": "usd"}'}]}

join orders -> customers: 'Each order has one customer.'
join orders -> products: None

measures on orders: ['total_amount', 'order_count', 'aov', 'revenue_line', 'cust_reach', 'rev_plus_pop', 'bridge_metric']


## Query the imported models

Point a SLayer client at the freshly imported models. Every query below is plain SLayer JSON — the OSI origin is now invisible.

In [5]:
client = SlayerClient(storage=YAMLStorage(base_dir=str(MODELS_DIR)))


def approx(a, b, tol=1e-9):
    return abs(a - b) < tol


def show(got, exp):
    """Print a SLayer scalar answer next to its gold value."""
    print("SLayer:", got, " gold:", exp)

### Query 1 — total order value

A simple metric: `total_amount` is `SUM(amount)`, imported from the OSI `SUM(amount)` expression as the SLayer formula `amount:sum`.

In [6]:
q1 = {"source_model": "orders", "measures": ["total_amount"]}
rows = client.query_sync(q1).data
print("SLayer:", rows)

assert approx(rows[0]["orders.total_amount"], GOLD["total"])
print(f"OK — SLayer and gold agree: {GOLD['total']}")

SLayer: [{'orders.total_amount': 1400.0}]
OK — SLayer and gold agree: 1400.0


### Query 2 — total by region (multi-hop join)

`total_amount` grouped by `customers.regions.name` walks two joins the converter inferred from OSI relationships — `orders -> customers -> regions` — with no SQL written by hand.

In [7]:
q2 = {
    "source_model": "orders",
    "measures": ["total_amount"],
    "dimensions": ["customers.regions.name"],
    "order": [{"column": "customers.regions.name"}],
}
rows = client.query_sync(q2).data
display(pd.DataFrame(rows))

slayer_pairs = [(r["orders.customers.regions.name"], r["orders.total_amount"]) for r in rows]
gold_pairs = [(g["region"], g["amount"]) for g in GOLD["by_region"]]
assert slayer_pairs == gold_pairs, f"{slayer_pairs} != {gold_pairs}"
print(f"OK — SLayer and gold agree on {len(slayer_pairs)} region(s)")

,orders.customers.regions.name,orders.total_amount
0,North,750.0
1,South,650.0


OK — SLayer and gold agree on 2 region(s)


### Query 3 — average order value (derived metric)

`aov` came from the OSI expression `(SUM(amount)) / (COUNT(*))` — a **derived** metric over two aggregates, imported as the formula `amount:sum / *:count`.

In [8]:
q3 = {"source_model": "orders", "measures": ["aov"]}
rows = client.query_sync(q3).data
show(rows[0]["orders.aov"], GOLD["aov"])

assert approx(rows[0]["orders.aov"], GOLD["aov"])
print("OK — average order value matches gold")

SLayer: 233.33333333333334  gold: 233.33333333333334
OK — average order value matches gold


### Query 4 — amount per distinct customer (cross-dataset metric)

`cust_reach` = `SUM(amount) / COUNT(DISTINCT customers.customer_id)`. The metric references a column on **another** dataset (`customers`); the converter anchored it on `orders` and emits the cross-model reference through the inferred join.

In [9]:
q4 = {"source_model": "orders", "measures": ["cust_reach"]}
rows = client.query_sync(q4).data
show(rows[0]["orders.cust_reach"], GOLD["cust_reach"])

assert approx(rows[0]["orders.cust_reach"], GOLD["cust_reach"])
print("OK — amount per distinct customer matches gold")

SLayer: 466.6666666666667  gold: 466.6666666666667
OK — amount per distinct customer matches gold


### Query 5 — amount plus region population (multi-hop metric)

`rev_plus_pop` = `SUM(amount) + SUM(regions.population)` reaches `regions` two joins away (`orders -> customers -> regions`).

The subtle part: SLayer aggregates `regions.population` **at the regions grain** (each distinct region counted once), so adding this measure can't multiply the order rows. A naive five-way join would instead add a region's population *once per order* in that region and badly over-count. Our gold mirrors SLayer's isolation — summing population over the distinct regions the orders reach — and the two agree.

In [10]:
q5 = {"source_model": "orders", "measures": ["rev_plus_pop"]}
rows = client.query_sync(q5).data
show(rows[0]["orders.rev_plus_pop"], GOLD["rev_plus_pop"])

assert approx(rows[0]["orders.rev_plus_pop"], GOLD["rev_plus_pop"])
print("OK — SLayer isolates the joined aggregate; both agree at", GOLD["rev_plus_pop"])

SLayer: 4400.0  gold: 4400.0
OK — SLayer isolates the joined aggregate; both agree at 4400.0


## Recap

Starting from an OSI config we did not write, SLayer:

- imported each `dataset` as a queryable model, with real column types from live introspection,
- turned OSI `relationships` into joins, so measures and dimensions reach across them (including multi-hop),
- folded simple, derived, cross-dataset, and multi-hop `metrics` into `ModelMeasure` formulas,
- and preserved `ai_context` / `custom_extensions` as descriptions and `meta` for agents to read.

Every answer matched gold SQL — including the multi-hop measure, where SLayer's sub-query isolation kept the added aggregate from fanning out the order rows.

### Further reading

- [Importing OSI configs](../../osi/osi_import.md) — the full conversion reference, including exactly what converts and what fails cleanly.
- The one-liner this notebook mirrors: `slayer import-osi shop.osi.yaml --datasource shop_osi` (against a registered datasource).